# Part 4: Curriculum Prerequisite Knowledge Graph & Cognitive Deficit Diagnosis
**AI-Powered Agentic Smart Campus Intelligence & Student Success Platform (PACD-RAG)**  
*Author: Sumith U (SRN: R24EA807) | 7th Sem B.Tech CSE (AI & ML), REVA University*

---

### Objectives:
1. Construct a formal Directed Acyclic Graph (DAG) of curriculum concepts using NetworkX.
2. Validate structural integrity (zero cycles, topological sortability).
3. Verify the core prerequisite sequence: *Partial Derivatives* $\to$ *Chain Rule* $\to$ *Backpropagation* $\to$ *Neural Network Training*.
4. Map student formative assessment evidence to concept nodes.
5. Trace root-cause prerequisite gaps when a student struggles with a target concept.
6. Generate actionable remediation payloads for downstream RAG (Part 5) and AI Agentic Coaching (Part 6).

In [ ]:
import os
import sys
import json
import networkx as nx
import matplotlib.pyplot as plt

# Add src/ to path
sys.path.append(os.path.abspath("../src"))
from prerequisite_graph import PrerequisiteKnowledgeGraph

# Initialize graph from data catalog
catalog_path = "../data/concept_prerequisites.json"
pkg = PrerequisiteKnowledgeGraph(catalog_path)
print("Knowledge graph initialized.")
print("Concepts:", pkg.graph.number_of_nodes())
print("Prerequisite Edges:", pkg.graph.number_of_edges())

### 1. Structural Validation of the Prerequisite DAG
Check for cycles, unknown endpoints, and confirm existence of a valid topological learning order.

In [ ]:
validation = pkg.validate_graph()
print("Graph Validation Results:")
print("  - Is Valid DAG:", validation["is_valid_dag"])
print("  - Has Cycles:", validation["has_cycles"])
print("  - Isolated Nodes:", len(validation["isolated_concepts"]))
print("  - Concepts Count:", validation["num_nodes"])
print("  - Dependencies Count:", validation["num_edges"])

print("\nFirst 8 Concepts in Recommended Topological Learning Order:")
for idx, cid in enumerate(validation["topological_order"][:8], 1):
    c_meta = pkg.concept_catalog[cid]
    print(f"  {idx}. [{cid}] {c_meta['name']} ({c_meta['unit']})")

### 2. Verification of the Key Pedagogical Chain
Validating the sequence specified in Part 4 requirements:  
**Partial Derivatives** $\to$ **Chain Rule** $\to$ **Backpropagation** $\to$ **Neural Network Training**

In [ ]:
chain = ["MATH_DERIV", "MATH_CHAIN_RULE", "DL_BACKPROP", "DL_NN_TRAIN"]
print("Verifying Prerequisite Chain:")
for i in range(len(chain) - 1):
    src = chain[i]
    tgt = chain[i + 1]
    is_edge = pkg.graph.has_edge(src, tgt)
    print(f"  {pkg.concept_catalog[src]['name']} -> {pkg.concept_catalog[tgt]['name']}: Edge exists? {is_edge}")

print("\nAll Ancestors of 'Neural Network Training':")
ancestors = pkg.get_all_prerequisites("DL_NN_TRAIN")
for a in sorted(list(ancestors)):
    print(f"  * [{a}] {pkg.concept_catalog[a]['name']}")

### 3. Student Formative Evidence Mapping & Prerequisite Gap Tracing
Simulate a student attempting an assessment in **Neural Network Training** (`DL_NN_TRAIN`) with unmastered prerequisite gaps.

In [ ]:
# Student assessment scores (0.0 to 1.0)
student_scores = {
    "MATH_DERIV": 0.88,         # Proficient
    "MATH_CHAIN_RULE": 0.44,    # Weak (below 0.60)
    "DL_FEEDFORWARD": 0.75,     # Proficient
    "DL_BACKPROP": 0.48,        # Weak (below 0.60)
    "ML_DEC_TREE": 0.30         # Weak, but completely unrelated to neural networks!
}

# Map evidence using non-punitive threshold (0.60)
diag = pkg.map_learner_evidence(student_scores, review_threshold=0.60)
weak_ids = [c["id"] for c in diag["needs_review_concepts"]]
print("Concepts needing review:", weak_ids)

# Trace causal prerequisite gaps for target concept 'DL_NN_TRAIN'
gap_analysis = pkg.trace_prerequisite_gaps("DL_NN_TRAIN", weak_ids)

print(f"\nTarget Learning Concept: {gap_analysis['target_concept']['name']}")
print("Diagnosed Causal Prerequisite Gaps:")
for step in gap_analysis["remediation_pathway"]:
    print(f"  Step {step['step']}: [{step['concept_id']}] {step['name']} ({step['unit']})")

print(f"\nUnrelated Weak Concepts Safely Filtered Out: {gap_analysis['unrelated_weak_concepts']}")
print(f"\nPedagogical Guidance: {gap_analysis['pedagogical_explanation']}")

### 4. Graph Visualization
Exporting high-resolution publication-quality visual DAG with highlighted remedial pathways.

In [ ]:
fig_path = "../reports/figures/prerequisite_knowledge_graph.png"
pkg.plot_graph(fig_path, highlight_target="DL_NN_TRAIN", highlight_weak=weak_ids)
print(f"Visual knowledge graph exported to: {fig_path}")

### 5. Downstream Integration Payloads (Part 5 RAG & Part 6 Multi-Agent)
Demonstrating the structured interfaces consumed by the RAG retriever and autonomous AI Student Coach.

In [ ]:
# Structured RAG Retrieval Plan (Part 5)
rag_plan = pkg.get_rag_retrieval_plan("DL_NN_TRAIN", weak_ids)
print("=== RAG RETRIEVAL PLAN (PART 5) ===")
print(json.dumps(rag_plan, indent=2))

# Structured Multi-Agent Coach Payload (Part 6)
coach_payload = pkg.get_agent_coach_payload("R24EA807_DEMO", "DL_NN_TRAIN", weak_ids)
print("\n=== AI STUDENT COACH PAYLOAD (PART 6) ===")
print(json.dumps(coach_payload, indent=2))